In [1]:
# ---- Kaggle GPU setup ----
# Make sure the GPU accelerator is turned on: Notebook Settings (right panel) -> Accelerator -> GPU T4 x2 (or P100).
# Also turn Internet ON in Notebook Settings, since this notebook downloads LibriSpeech/MUSAN and pip-installs packages.
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — go to Settings > Accelerator and enable a GPU, then re-run.")

CUDA available: True
GPU: Tesla T4


In [2]:
!find /kaggle/input -maxdepth 3 -type d

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/cs25m115raushanvivek
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice


In [3]:
import os
from pathlib import Path

DATA_ROOT = '/kaggle/working/data'
CACHE_DIR = '/kaggle/working/data'  # replaces the old Google Drive cache; /kaggle/working persists as the notebook's output
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(f'{CACHE_DIR}/checkpoints', exist_ok=True)

In [4]:
!echo "--- /kaggle/working disk usage ---"
!du -sh /kaggle/working/* 2>/dev/null
!echo "--- overall usage on /kaggle/working (20GB quota) ---"
!df -h /kaggle/working

--- /kaggle/working disk usage ---
8.0K	/kaggle/working/data
--- overall usage on /kaggle/working (20GB quota) ---
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   88K   20G   1% /kaggle/working


In [5]:
INPUT_DIR = "/kaggle/input/datasets/cs25m115raushanvivek/fake-voice"

# Read directly from the read-only input mount — no need to copy it into /kaggle/working,
# which only wastes space on the 20GB output quota. clean_dataset() only reads these files.
fake_raw_dir = f"{INPUT_DIR}/dataset-demo-bucket-26"

!find {fake_raw_dir} -type f | head
!find {fake_raw_dir} -type f | wc -l

/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_761.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_138.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_890.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_646.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_244.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_650.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_914.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_719.wav
/kaggle/input/datasets/cs25m115raushanvivek/fake-voice/dataset-demo-bucket-26/en/eleven_v3/eleven_v3_161.wav
/kaggle/input/datas

In [6]:
raw_fake=fake_raw_dir

Download LibriSpeech

In [7]:
# Download + extract LibriSpeech in one streamed step (wget | tar), so the ~6.3GB
# compressed archive is never written to disk — only the extracted FLACs are, and even
# those get deleted right after cleaning (see below). This avoids double-storing the data.
real_raw_dir = f"{DATA_ROOT}/raw_real"
os.makedirs(real_raw_dir, exist_ok=True)

if not any(Path(real_raw_dir).rglob("*.flac")):
    !wget -qO- https://www.openslr.org/resources/12/train-clean-100.tar.gz | tar xz -C {real_raw_dir}
else:
    print("Already extracted, skipping.")

# Count FLAC files
!find {real_raw_dir} -name "*.flac" | wc -l

28539


In [8]:
# No-op now: the streamed download above never wrote train-clean-100.tar.gz to disk,
# so there's nothing to delete here.

In [9]:
# !rm -rf /kaggle/working/data/

In [10]:
# MUSAN's music/ and speech/ subfolders (~9GB combined) are never used below — only
# musan/noise/ is. Streaming the download straight into tar with a wildcard filter means
# we extract (and pay disk quota for) only the ~1-2GB noise subset, and never write the
# full ~11GB .tar.gz to disk either.
musan_dir = f'{DATA_ROOT}/musan'
os.makedirs(musan_dir, exist_ok=True)

noise_dir = Path(musan_dir) / "musan" / "noise"
if not noise_dir.exists() or not any(noise_dir.rglob("*.wav")):
    !wget -qO- https://www.openslr.org/resources/17/musan.tar.gz | tar xz -C {musan_dir} --wildcards 'musan/noise/*'
else:
    print('Already extracted, skipping.')

!find {musan_dir}/musan/noise -name '*.wav' | wc -l

930


#Data cleaning functions
Handles: corrupt files, silence trimming, non-speech rejection (VAD), consistent sample rate.
Cleaned audio is written to `data/clean_fake/` and `data/clean_real/` at 16kHz.

In [11]:
!pip install webrtcvad

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73597 sha256=87084570eadd7a87092d6d7723b4fbddfb296ec3ff43dd03ee63e3402bae30a0
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614
Successfully built webrtcvad


In [12]:
import numpy as np
import librosa
import soundfile as sf
import webrtcvad
import struct

TARGET_SR = 16000
MIN_DURATION_SEC = 1.0     # reject clips shorter than this after trimming
MAX_DURATION_SEC = 15.0    # cap very long clips (keeps training clip-friendly)
MIN_SPEECH_RATIO = 0.3     # reject clips where VAD detects speech in <30% of frames

vad = webrtcvad.Vad(2)  # aggressiveness 0-3; 2 is a reasonable middle ground

def load_audio_safely(path, target_sr=TARGET_SR):
    """Returns (wav, sr) or (None, None) if the file is corrupt/unreadable."""
    try:
        wav, sr = librosa.load(path, sr=target_sr, mono=True)
        if wav is None or len(wav) == 0 or not np.isfinite(wav).all():
            return None, None
        return wav, sr
    except Exception:
        return None, None

def trim_silence(wav, sr, top_db=30):
    trimmed, _ = librosa.effects.trim(wav, top_db=top_db)
    return trimmed

def vad_speech_ratio(wav, sr=TARGET_SR, frame_ms=30):
    """Fraction of frames classified as speech by WebRTC VAD.
    webrtcvad requires 16-bit PCM mono at 8/16/32/48kHz, frame len 10/20/30ms."""
    pcm16 = (wav * 32767).astype(np.int16).tobytes()
    frame_len = int(sr * frame_ms / 1000) * 2  # *2 bytes per int16 sample
    n_frames = len(pcm16) // frame_len
    if n_frames == 0:
        return 0.0
    speech_frames = 0
    for i in range(n_frames):
        frame = pcm16[i * frame_len:(i + 1) * frame_len]
        if len(frame) < frame_len:
            break
        try:
            if vad.is_speech(frame, sr):
                speech_frames += 1
        except Exception:
            continue
    return speech_frames / n_frames

def clean_one_file(in_path, out_path):
    """Full cleaning chain. Returns metadata dict, or None if rejected."""
    wav, sr = load_audio_safely(in_path)
    if wav is None:
        return None  # corrupt

    wav = trim_silence(wav, sr)
    duration = len(wav) / sr
    if duration < MIN_DURATION_SEC:
        return None  # too short after trimming (likely all silence)

    if duration > MAX_DURATION_SEC:
        wav = wav[: int(MAX_DURATION_SEC * sr)]
        duration = MAX_DURATION_SEC

    speech_ratio = vad_speech_ratio(wav, sr)
    if speech_ratio < MIN_SPEECH_RATIO:
        return None  # non-speech / mostly noise

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    sf.write(out_path, wav, sr)
    return {"filepath": out_path, "duration": duration, "speech_ratio": speech_ratio}

/usr/local/lib/python3.12/dist-packages/webrtcvad.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


#Run cleaning over both datasets
applies the cleaning chain, and records what got kept vs rejected and why.

In [13]:
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import os

def clean_dataset(src_dir, dst_dir, label, source_name,
                  speaker_from_path=None,
                  exts=("*.wav", "*.flac", "*.mp3"),
                  delete_source=False):

    rows = []
    rejected = {"corrupt_or_silent": 0, "non_speech": 0}
    files = []

    for ext in exts:
        files.extend(Path(src_dir).rglob(ext))

    for f in tqdm(files, desc=f"cleaning {source_name}"):

        out_path = Path(dst_dir) / f.relative_to(src_dir).with_suffix(".wav")

        meta = clean_one_file(str(f), str(out_path))

        if meta is None:
            rejected["corrupt_or_silent"] += 1
            continue

        # Delete original file after successful processing
        if delete_source:
            try:
                os.remove(f)
            except Exception as e:
                print(f"Couldn't delete {f}: {e}")

        speaker_id = speaker_from_path(f) if speaker_from_path else f.stem

        rows.append({
            "filepath": meta["filepath"],
            "label": label,
            "speaker_id": speaker_id,
            "source_dataset": source_name,
            "duration": meta["duration"],
        })

    print(f"[{source_name}] kept {len(rows)} / {len(files)} files; rejected {rejected}")
    return pd.DataFrame(rows)

def librispeech_speaker_id(filepath):
    # LibriSpeech layout: .../<speaker>/<chapter>/<speaker>-<chapter>-<utt>.flac
    return filepath.parent.parent.name

clean_fake_dir = f'{DATA_ROOT}/clean_fake'
clean_real_dir = f'{DATA_ROOT}/clean_real'

fake_df = clean_dataset(fake_raw_dir, clean_fake_dir, label=1, source_name="provided_fake")

real_df = clean_dataset(real_raw_dir, clean_real_dir, label=0, source_name="librispeech",
                         speaker_from_path=librispeech_speaker_id, exts=("*.flac",))
!rm -rf /kaggle/working/data/raw_real
print(f"\nTotal fake duration: {fake_df['duration'].sum()/3600:.2f} hours")
print(f"Total real duration: {real_df['duration'].sum()/3600:.2f} hours")

cleaning provided_fake: 100%|██████████| 11985/11985 [03:39<00:00, 54.68it/s]


[provided_fake] kept 11939 / 11985 files; rejected {'corrupt_or_silent': 46, 'non_speech': 0}


cleaning librispeech: 100%|██████████| 28539/28539 [06:58<00:00, 68.15it/s]


[librispeech] kept 28538 / 28539 files; rejected {'corrupt_or_silent': 1, 'non_speech': 0}

Total fake duration: 17.77 hours
Total real duration: 96.20 hours


##Balance real vs fake sensibly
Balances by **total duration**, not file count (a fair comparison since clip lengths vary).
Downsamples the larger class to match the smaller class's total hours, sampling at the **speaker level** for real audio so we don't break a speaker across the cut.

In [24]:
import random
random.seed(42)

def balance_by_duration(fake_df, real_df, seed=42):
    fake_hours = fake_df['duration'].sum() / 3600
    real_hours = real_df['duration'].sum() / 3600
    target_hours = min(fake_hours, real_hours)
    print(f"Balancing both classes down to ~{target_hours:.2f} hours each")

    def cap_by_duration(df, target_seconds, by_speaker=False):
        rng = random.Random(seed)
        if by_speaker:
            speakers = df['speaker_id'].unique().tolist()
            rng.shuffle(speakers)
            kept_rows, total = [], 0.0
            for spk in speakers:
                spk_rows = df[df['speaker_id'] == spk]
                kept_rows.append(spk_rows)
                total += spk_rows['duration'].sum()
                if total >= target_seconds:
                    break
            return pd.concat(kept_rows, ignore_index=True)
        else:
            idx = df.index.tolist()
            rng.shuffle(idx)
            kept_idx, total = [], 0.0
            for i in idx:
                kept_idx.append(i)
                total += df.loc[i, 'duration']
                if total >= target_seconds:
                    break
            return df.loc[kept_idx].reset_index(drop=True)

    target_seconds = target_hours * 3600
    fake_balanced = cap_by_duration(fake_df, target_seconds, by_speaker=False)
    real_balanced = cap_by_duration(real_df, target_seconds, by_speaker=True)
    return fake_balanced, real_balanced

fake_balanced, real_balanced = balance_by_duration(fake_df, real_df)
print(f"Fake: {len(fake_balanced)} files, {fake_balanced['duration'].sum()/3600:.2f}h")
print(f"Real: {len(real_balanced)} files, {real_balanced['duration'].sum()/3600:.2f}h")

# Free space: once we've balanced by duration, the files that didn't make the cut
# are no longer needed by anything downstream (train/val/test all come from the
# balanced+split manifest). Deleting them reclaims a meaningful chunk of the 20GB quota.
def remove_unused_files(full_df, kept_df):
    kept_paths = set(kept_df['filepath'])
    removed = 0
    for p in full_df['filepath']:
        if p not in kept_paths:
            try:
                os.remove(p)
                removed += 1
            except FileNotFoundError:
                pass
    print(f"Removed {removed} unused cleaned files")

remove_unused_files(fake_df, fake_balanced)
remove_unused_files(real_df, real_balanced)

Balancing both classes down to ~17.77 hours each
Fake: 11939 files, 17.77h
Real: 5273 files, 17.88h
Removed 0 unused cleaned files
Removed 0 unused cleaned files


In [15]:
!echo "--- /kaggle/working disk usage ---"
!du -sh /kaggle/working/* 2>/dev/null
!echo "--- overall usage on /kaggle/working (20GB quota) ---"
!df -h /kaggle/working

# Compare this against the earlier check — you should see clean_fake/clean_real shrink.

--- /kaggle/working disk usage ---
4.6G	/kaggle/working/data
--- overall usage on /kaggle/working (20GB quota) ---
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  4.6G   15G  24% /kaggle/working


#Speaker-disjoint train/val/test split + leakage check

In [16]:
def speaker_disjoint_split(df, train_frac=0.70, val_frac=0.15, seed=42):
    rng = random.Random(seed)
    speakers = df['speaker_id'].unique().tolist()
    rng.shuffle(speakers)
    n = len(speakers)
    n_train = int(n * train_frac)
    n_val = int(n * val_frac)
    train_spk = set(speakers[:n_train])
    val_spk = set(speakers[n_train:n_train + n_val])

    def assign(spk):
        if spk in train_spk: return 'train'
        elif spk in val_spk: return 'val'
        else: return 'test'

    df = df.copy()
    df['split'] = df['speaker_id'].apply(assign)
    return df

# Modify fake_balanced speaker_ids to prevent collision with real_balanced speaker_ids
fake_balanced_modified = fake_balanced.copy()
fake_balanced_modified['speaker_id'] = fake_balanced_modified['speaker_id'].apply(lambda x: f"fake_{x}")

fake_split = speaker_disjoint_split(fake_balanced_modified)   # fake files have unique per-file "speaker_id" (filename), so this is file-disjoint
real_split = speaker_disjoint_split(real_balanced)   # real files split by true LibriSpeech speaker ID

manifest = pd.concat([fake_split, real_split], ignore_index=True)

# Leakage assertions
for key in ('speaker_id', 'filepath'):
    counts = manifest.groupby(key)['split'].nunique()
    leaked = counts[counts > 1]
    assert len(leaked) == 0, f"Leakage in {key}: {leaked}"
print("[ok] no leakage across splits")

manifest_path = f'{CACHE_DIR}/manifest.csv'
# Ensure the directory exists before saving the CSV
os.makedirs(os.path.dirname(manifest_path), exist_ok=True)
manifest.to_csv(manifest_path, index=False)
print(manifest.groupby(['split', 'label']).size())
print(f"\nSaved manifest to {manifest_path}")

[ok] no leakage across splits
split  label
test   0         952
       1        1814
train  0        3548
       1        8343
val    0         773
       1        1782
dtype: int64

Saved manifest to /kaggle/working/data/manifest.csv


#VoIP degradation pipeline (for training augmentation + eval)

In [17]:
import subprocess, tempfile, random as _random

def _run_ffmpeg(cmd):
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def opus_roundtrip(wav, sr):
    with tempfile.TemporaryDirectory() as td:
        in_p, opus_p, out_p = f'{td}/in.wav', f'{td}/o.opus', f'{td}/out.wav'
        sf.write(in_p, wav, sr)
        _run_ffmpeg(['ffmpeg','-y','-i',in_p,'-c:a','libopus','-b:a','24k',opus_p])
        _run_ffmpeg(['ffmpeg','-y','-i',opus_p,'-ar',str(sr),out_p])
        out, _ = sf.read(out_p)
        return out.astype(np.float32)

def g711_roundtrip(wav, sr):
    with tempfile.TemporaryDirectory() as td:
        in_p, mu_p, out_p = f'{td}/in.wav', f'{td}/mu.wav', f'{td}/out.wav'
        sf.write(in_p, wav, sr)
        _run_ffmpeg(['ffmpeg','-y','-i',in_p,'-ar','8000','-acodec','pcm_mulaw',mu_p])
        _run_ffmpeg(['ffmpeg','-y','-i',mu_p,'-ar',str(sr),out_p])
        out, _ = sf.read(out_p)
        return out.astype(np.float32)

def downsample_roundtrip(wav, sr, target_hz=8000):
    down = librosa.resample(wav, orig_sr=sr, target_sr=target_hz)
    back = librosa.resample(down, orig_sr=target_hz, target_sr=sr)
    if len(back) < len(wav):
        back = np.pad(back, (0, len(wav) - len(back)))
    return back[:len(wav)].astype(np.float32)

def add_noise(wav, noise, snr_db):
    if len(noise) < len(wav):
        noise = np.tile(noise, int(np.ceil(len(wav)/len(noise))))
    noise = noise[:len(wav)]
    sig_p = np.mean(wav**2) + 1e-12
    noise_p = np.mean(noise**2) + 1e-12
    scale = np.sqrt((sig_p / (10**(snr_db/10))) / noise_p)
    return (wav + scale*noise).astype(np.float32)

def degrade(wav, sr, noise_bank=None, codec_choices=("opus","g711"), snr_choices=(0,5,10,15)):
    codec = _random.choice(codec_choices)
    wav = opus_roundtrip(wav, sr) if codec == "opus" else g711_roundtrip(wav, sr)
    wav = downsample_roundtrip(wav, sr)
    if noise_bank:
        wav = add_noise(wav, _random.choice(noise_bank), _random.choice(snr_choices))
    return wav

# preload a noise bank once
noise_files = list(Path(f'{musan_dir}/musan/noise').rglob('*.wav'))[:200]
noise_bank = [librosa.load(f, sr=TARGET_SR)[0].astype(np.float32) for f in tqdm(noise_files, desc='loading noise bank')]
print(f"Loaded {len(noise_bank)} noise files")

loading noise bank: 100%|██████████| 200/200 [00:01<00:00, 176.90it/s]

Loaded 200 noise files


# Dataset + feature extraction

In [18]:
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

CLIP_SECONDS = 4.0
TARGET_LEN = int(TARGET_SR * CLIP_SECONDS)
N_MELS = 80

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR, n_fft=400, win_length=400, hop_length=160, n_mels=N_MELS)
db_transform = torchaudio.transforms.AmplitudeToDB()

def extract_logmel(wav_t):
    if wav_t.dim() == 1:
        wav_t = wav_t.unsqueeze(0)
    return db_transform(mel_transform(wav_t)).squeeze(0)

def crop_or_pad(wav_t, target_len=TARGET_LEN):
    n = wav_t.shape[-1]
    if n == target_len:
        return wav_t
    if n > target_len:
        start = torch.randint(0, n - target_len + 1, (1,)).item()
        return wav_t[start:start+target_len]
    return torch.nn.functional.pad(wav_t, (0, target_len - n))

class DeepfakeDataset(Dataset):
    def __init__(self, manifest_df, split, augment=False, augment_prob=0.5):
        self.df = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.augment = augment
        self.augment_prob = augment_prob

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, sr = librosa.load(row['filepath'], sr=TARGET_SR)
        wav = wav.astype(np.float32)
        if self.augment and _random.random() < self.augment_prob:
            wav = degrade(wav, TARGET_SR, noise_bank=noise_bank)
        wav_t = crop_or_pad(torch.from_numpy(wav))
        feats = extract_logmel(wav_t)
        return {"features": feats, "label": int(row['label'])}

train_ds = DeepfakeDataset(manifest, 'train', augment=True, augment_prob=0.5)
val_clean_ds = DeepfakeDataset(manifest, 'val', augment=False)
val_voip_ds = DeepfakeDataset(manifest, 'val', augment=True, augment_prob=1.0)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_clean_loader = DataLoader(val_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
val_voip_loader = DataLoader(val_voip_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"train={len(train_ds)} val_clean={len(val_clean_ds)} val_voip={len(val_voip_ds)}")

train=11891 val_clean=2555 val_voip=2555


#Model — ECAPA-TDNN + attentive pooling + FC head

In [19]:
!pip install speechbrain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 43.5 MB/s eta 0:00:00
  Attempting uninstall: ruamel.yaml
    Found existing installation: ruamel.yaml 0.19.1
    Uninstalling ruamel.yaml-0.19.1:
      Successfully uninstalled ruamel.yaml-0.19.1


In [20]:
import torch.nn as nn
from speechbrain.lobes.models.ECAPA_TDNN import ECAPA_TDNN

class DeepfakeDetector(nn.Module):
    def __init__(self, n_mels=80, channels=512, embedding_dim=192, dropout=0.3, num_classes=2):
        super().__init__()
        self.backbone = ECAPA_TDNN(
            input_size=n_mels,
            channels=[channels]*4 + [channels*3],
            kernel_sizes=[5,3,3,3,1],
            dilations=[1,2,3,4,1],
            attention_channels=128,
            lin_neurons=embedding_dim,
        )
        self.head = nn.Sequential(
            nn.Linear(embedding_dim, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, num_classes)
        )

    def forward(self, feats):
        x = feats.transpose(1, 2)
        emb = self.backbone(x).squeeze(1)
        return self.head(emb)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepfakeDetector().to(device)
print(f"Model on {device}, params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

Model on cuda, params: 6.22M


#Training loop (checkpoints on val-VoIP EER, not val-clean)

In [21]:
from sklearn.metrics import roc_auc_score, roc_curve

NUM_EPOCHS = 30
PATIENCE = 8
LR = 1e-3

def compute_eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2

@torch.no_grad()
def evaluate_split(model, loader):
    model.eval()
    all_labels, all_scores = [], []
    for batch in loader:
        feats = batch['features'].to(device)
        logits = model(feats)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_labels.extend(batch['label'].numpy())
        all_scores.extend(probs)
    all_labels, all_scores = np.array(all_labels), np.array(all_scores)
    eer = compute_eer(all_labels, all_scores)
    auc = roc_auc_score(all_labels, all_scores)
    preds = (all_scores >= 0.5).astype(int)
    acc = (preds == all_labels).mean()
    tp = ((preds==1)&(all_labels==1)).sum(); fp = ((preds==1)&(all_labels==0)).sum(); fn = ((preds==0)&(all_labels==1)).sum()
    prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
    f1 = 2*prec*rec/(prec+rec+1e-9)
    return {"eer": eer, "auc": auc, "accuracy": acc, "f1": f1}

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
total_steps = max(NUM_EPOCHS * len(train_loader), 1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=total_steps, pct_start=0.05)
criterion = nn.CrossEntropyLoss()

best_val_voip_eer = float('inf')
epochs_no_improve = 0
ckpt_path = f'{CACHE_DIR}/checkpoints/best.pt'

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for batch in tqdm(train_loader, desc=f"epoch {epoch+1}/{NUM_EPOCHS}"):
        feats = batch['features'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    val_clean = evaluate_split(model, val_clean_loader)
    val_voip = evaluate_split(model, val_voip_loader)
    print(f"[epoch {epoch+1}] loss={running_loss/len(train_loader):.4f} "
          f"val_clean_eer={val_clean['eer']:.4f} val_voip_eer={val_voip['eer']:.4f}")

    if val_voip['eer'] < best_val_voip_eer:
        best_val_voip_eer = val_voip['eer']
        epochs_no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print(f"[ok] new best checkpoint saved (val_voip_eer={best_val_voip_eer:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"[info] early stopping at epoch {epoch+1}")
            break

epoch 1/30: 100%|██████████| 372/372 [18:01<00:00,  2.91s/it]


[epoch 1] loss=0.3412 val_clean_eer=0.1800 val_voip_eer=0.2960
[ok] new best checkpoint saved (val_voip_eer=0.2960)


epoch 2/30: 100%|██████████| 372/372 [18:15<00:00,  2.94s/it]


[epoch 2] loss=0.2987 val_clean_eer=0.1200 val_voip_eer=0.2710
[ok] new best checkpoint saved (val_voip_eer=0.2710)


epoch 3/30: 100%|██████████| 372/372 [18:05<00:00,  2.92s/it]


...

[epoch 14] loss=0.1290 val_clean_eer=0.1077 val_voip_eer=0.2386


epoch 15/30: 100%|██████████| 372/372 [18:33<00:00,  2.99s/it]


[epoch 15] loss=0.1190 val_clean_eer=0.0463 val_voip_eer=0.2119


epoch 16/30: 100%|██████████| 372/372 [17:37<00:00,  2.84s/it]


[epoch 16] loss=0.1173 val_clean_eer=0.0517 val_voip_eer=0.2206
[info] early stopping at epoch 16


#Final evaluation on held-out TEST set (clean + VoIP-degraded) -> results.md

In [22]:
test_clean_ds = DeepfakeDataset(manifest, 'test', augment=False)
test_voip_ds = DeepfakeDataset(manifest, 'test', augment=True, augment_prob=1.0)
test_clean_loader = DataLoader(test_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_voip_loader = DataLoader(test_voip_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model.load_state_dict(torch.load(ckpt_path, map_location=device))
clean_metrics = evaluate_split(model, test_clean_loader)
voip_metrics = evaluate_split(model, test_voip_loader)

results_md = f"""# Results

| Condition | EER | AUC | Accuracy | F1 |
|---|---|---|---|---|
| Clean | {clean_metrics['eer']:.4f} | {clean_metrics['auc']:.4f} | {clean_metrics['accuracy']:.4f} | {clean_metrics['f1']:.4f} |
| VoIP-degraded (codec + noise) | {voip_metrics['eer']:.4f} | {voip_metrics['auc']:.4f} | {voip_metrics['accuracy']:.4f} | {voip_metrics['f1']:.4f} |

## Data
- Fake: provided ~40h set, cleaned (corrupt/silence/non-speech removed), balanced by duration.
- Real: LibriSpeech train-clean-100, cleaned + balanced by duration, speaker-disjoint split.

## VoIP degradation
Random Opus/G.711 codec round-trip + 8kHz downsample round-trip + MUSAN noise at random SNR in {{0,5,10,15}} dB.

## Threshold
0.5 on P(fake) (default). Consider reporting the val-set EER-point threshold for a stricter comparison.
"""

with open(f'{CACHE_DIR}/results.md', 'w') as f:
    f.write(results_md)
print(results_md)

# Results

| Condition | EER | AUC | Accuracy | F1 |
|---|---|---|---|---|
| Clean | 0.0296 | 0.9965 | 0.9729 | 0.9793 |
| VoIP-degraded (codec + noise) | 0.1482 | 0.9251 | 0.8568 | 0.8879 |

## Data
- Fake: provided ~40h set, cleaned (corrupt/silence/non-speech removed), balanced by duration.
- Real: LibriSpeech train-clean-100, cleaned + balanced by duration, speaker-disjoint split.

## VoIP degradation
Random Opus/G.711 codec round-trip + 8kHz downsample round-trip + MUSAN noise at random SNR in {0,5,10,15} dB.

## Threshold
0.5 on P(fake) (default). Consider reporting the val-set EER-point threshold for a stricter comparison.

